# Exploration lane — 3 idea probes (Colab RTX PRO 6000, **NOT paper-grade**)

Cheap probes vs CARD-PB (FLOPs 0.665, l1 holds). Signal only; Kiet reads `SUMMARY.md` + decides. Writes ONLY to Drive `results/exploration/`, tagged `exploration:true device:colab`. Uses your existing dataset at `/content/drive/MyDrive/attackdro/data` (no re-download). Pinned `1d901cad`.


## 1 · Setup — clone @ pin, install

In [ ]:
!git clone --quiet https://github.com/anhkiet287/attackdro.git 2>/dev/null || (cd attackdro && git fetch --quiet origin)
%cd attackdro
!git checkout --quiet 1d901cad
!pip install --quiet -e . 2>/dev/null || true
import os; os.environ['PYTHONPATH']='/content/attackdro/src'
!git log --oneline -1

## 2 · Drive + data (USE EXISTING dataset, no download) + guards

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, yaml, torch
DATA='/content/drive/MyDrive/attackdro/data'   # your existing CIFAR-10
EXPDIR='/content/drive/MyDrive/attackdro/results/exploration'; os.makedirs(EXPDIR, exist_ok=True)
assert os.path.isdir(f'{DATA}/cifar-10-batches-py'), (
    f'CIFAR-10 not found at {DATA}/cifar-10-batches-py — fix DATA or unpack the dataset there '
    f'(expected torchvision layout: {DATA}/cifar-10-batches-py/data_batch_1 ...)')
print('dataset OK:', os.listdir(f'{DATA}/cifar-10-batches-py')[:3])
tm=yaml.safe_load(open('configs/base.yaml'))['threat_model']
assert abs(tm['linf']['eps']-8/255)<1e-6 and tm['l2']['eps']==0.5 and tm['l1']['eps']==12, 'EPS GUARD FAILED'
print('eps OK:', tm, '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARN)')

## 3 · W&B (online; set disabled to skip)

In [ ]:
os.environ['WANDB_MODE']='online'
if os.environ['WANDB_MODE']=='online':
    try:
        import wandb; wandb.login()
    except Exception as e:
        print('wandb skipped:', e); os.environ['WANDB_MODE']='disabled'

## 4 · Idea 1 — fail-rate threshold allocation (ep20)

In [ ]:
!python scripts/train.py --config configs/exploration/idea1_failrate.yaml --seed 0 --run-name idea1_failrate --set results_dir={EXPDIR}/ --set dataset.root={DATA}
!python scripts/evaluate.py --config configs/exploration/idea1_failrate.yaml --checkpoint {EXPDIR}/idea1_failrate/s0/ckpt/last.pt --run-name idea1_failrate --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/ --set dataset.root={DATA}

## 5 · Idea 2 — k_min self-recovery (kspan=4, k_min=1, ep30)

In [ ]:
!python scripts/train.py --config configs/exploration/idea2_kmin_recovery.yaml --seed 0 --run-name idea2_kmin_recovery --set results_dir={EXPDIR}/ --set dataset.root={DATA}
!python scripts/evaluate.py --config configs/exploration/idea2_kmin_recovery.yaml --checkpoint {EXPDIR}/idea2_kmin_recovery/s0/ckpt/last.pt --run-name idea2_kmin_recovery --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/ --set dataset.root={DATA}

## 6 · Idea 3 — curriculum vs fixed-10 control (ep30 each)

In [ ]:
!python scripts/train.py --config configs/exploration/idea3_curriculum.yaml --seed 0 --run-name idea3_curriculum --set results_dir={EXPDIR}/ --set dataset.root={DATA}
!python scripts/evaluate.py --config configs/exploration/idea3_curriculum.yaml --checkpoint {EXPDIR}/idea3_curriculum/s0/ckpt/last.pt --run-name idea3_curriculum --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/ --set dataset.root={DATA}
!python scripts/train.py --config configs/exploration/idea3_control_fixed10.yaml --seed 0 --run-name idea3_control_fixed10 --set results_dir={EXPDIR}/ --set dataset.root={DATA}
!python scripts/evaluate.py --config configs/exploration/idea3_control_fixed10.yaml --checkpoint {EXPDIR}/idea3_control_fixed10/s0/ckpt/last.pt --run-name idea3_control_fixed10 --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/ --set dataset.root={DATA}

## 7 · Read → verdicts + SUMMARY

In [ ]:
!python scripts/dev/exploration_read.py {EXPDIR}
from IPython.display import Markdown, display
display(Markdown(open(f'{EXPDIR}/SUMMARY.md').read()))

---
**STOP.** Signal not paper numbers. Send `SUMMARY.md` to Kiet.